"""
This dashboard is part of a project where I explored employment patterns in Odisha
using PLFS 2024–2025 data.

The idea was to check whether there is a shift from agriculture to other sectors.
I processed the raw survey data, applied weights, and then built this dashboard
to make the results easier to understand.

The visualizations show:
- Sector-wise distribution
- Trend over time
- Differences across gender and rural/urban areas

Data Source:

PLFS (Periodic Labour Force Survey) – Government of India

Official Website:
https://mospi.gov.in

Download Link:
https://microdata.gov.in/nada43/index.php/catalog/PLFS

Files used:
- CHHV1 (Household data)
- CPERV1 (Person data)

Note:
2024 data is available in CSV format.
2025 data is in TXT format and parsed using layout specifications.
"""

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

Loading and Pre-Processing 2024 Data

In [ ]:
# ---------------------------
# Load 2024 PLFS data
# ---------------------------
# Person-level file (individual info)
df_perv_2024  = pd.read_csv("/data/PLFS 2024/cperv1.csv")

# Household-level file (household attributes)
df_hh_2024 = pd.read_csv("/data/PLFS 2024/chhv1.csv")


# ---------------------------
# Standardize column names
# ---------------------------
# Convert all column names to lowercase for consistency
df_perv_2024.columns = df_perv_2024.columns.str.lower()
df_hh_2024.columns = df_hh_2024.columns.str.lower()


# ---------------------------
# Rename columns (make them usable)
# ---------------------------
# Mapping raw PLFS column names to simpler names
df_perv_2024.rename(columns={
    'sector': 'sector',                     # Rural / Urban
    'state_ut_code': 'state',
    'district_code': 'district',
    'sex': 'gender',
    'age': 'age',

    # Using Current Weekly Status (CWS) variables
    'cws_industry_code': 'industry',        # NIC code
    'cws_status_code': 'worker_status',     # employment status

    'fsu': 'fsu',                           # sampling unit
    'sample_household_number': 'hh_id',     # household id
    'second_stage_stratum_no': 'sss',
    'subsample_multiplier': 'mult'          # survey weight
}, inplace=True)


df_hh_2024.rename(columns={
    'sector': 'sector',
    'state_ut_code': 'state',
    'district_code': 'district',
    'fsu': 'fsu',
    'sample_household_number': 'hh_id',
    'second_stage_stratum_no': 'sss',

    # Household-level features
    'household_size': 'hh_size',
    'household_type': 'hhtype',
    'social_group': 'sg',
    'religion': 'relg',
    'monthly_consumer_expenditure': 'hce_tot',

    'subsample_multiplier': 'mult'
}, inplace=True)


# ---------------------------
# Merge person + household data
# ---------------------------
# These keys uniquely identify a household in PLFS
merge_keys = ['sector','state','district','fsu','sss','hh_id']

df_2024 = df_perv_2024.merge(
    df_hh_2024,
    on=merge_keys,
    how='left',              # keep all individuals, attach household info
    suffixes=('_p','_hh')
)


# ---------------------------
# Handle duplicate weight column after merge
# ---------------------------
# Prefer person-level multiplier
if 'mult_p' in df_2024.columns:
    df_2024.rename(columns={'mult_p':'mult'}, inplace=True)


# ---------------------------
# Select only required columns
# ---------------------------
cols = [
    'sector','state','district',
    'gender','age','industry','worker_status','mult',
    'hh_size','hhtype','sg','relg','hce_tot'
]

# Keep only columns that actually exist (safe check)
cols = [c for c in cols if c in df_2024.columns]
df_2024 = df_2024[cols].copy()


# ---------------------------
# Convert all columns to numeric where possible
# ---------------------------
# Non-numeric values will become NaN
for c in df_2024.columns:
    df_2024[c] = pd.to_numeric(df_2024[c], errors='coerce')


# ---------------------------
# Keep only working population
# ---------------------------
# 11 = self-employed, 12 = salaried, 21 = casual labour
df_2024 = df_2024[df_2024['worker_status'].isin([11,12,21])]


# ---------------------------
# Drop invalid / missing data
# ---------------------------
# Ensure key variables are present
df_2024 = df_2024[df_2024['state'].notna() & df_2024['mult'].notna()]
df_2024 = df_2024[df_2024['industry'].notna()]


# ---------------------------
# Apply survey weights
# ---------------------------
# Convert multiplier to usable weight
df_2024['weight'] = df_2024['mult'] / 100

Loading 2025 PLFS Data

In [ ]:
# ---------------------------
# Function to build column specs from layout file
# ---------------------------
# PLFS 2025 data is in fixed-width format (TXT),
# so we need start & end byte positions for each column.

def build_colspecs(layout_df):
    colspecs, colnames = [], []

    # Iterate over each row of layout (each row = one column definition)
    for _, row in layout_df.iterrows():
        try:
            # Byte positions are 1-based in layout → convert to 0-based indexing
            start = int(row['Byte Position (Start)']) - 1
            end = int(row['Byte Position (End)'])

            # Clean column name
            name = str(row['Field_Name']).strip().lower()

            # Store positions + column names
            colspecs.append((start, end))
            colnames.append(name)

        except:
            # Skip rows where layout info is missing / invalid
            continue

    return colspecs, colnames


# ---------------------------
# Load layout file
# ---------------------------
# This Excel file tells us how to read the TXT files
layout = pd.read_excel(
    "/data/PLFS 2025/FV_Data_LayoutPLFS_2025.xlsx",
    sheet_name=None
)

# Separate layouts for person and household data
perv_layout = layout['CPERV1']
hh_layout = layout['CHHV1']


# ---------------------------
# Generate column specifications
# ---------------------------
# These will be used by read_fwf to correctly slice each column
colspecs_perv, colnames_perv = build_colspecs(perv_layout)
colspecs_hh, colnames_hh = build_colspecs(hh_layout)


# ---------------------------
# Read fixed-width files
# ---------------------------
# Using colspecs ensures correct parsing of each field

df_perv_2025 = pd.read_fwf(
    "/data/PLFS 2025/CPERV1.TXT",
    colspecs=colspecs_perv,
    names=colnames_perv
)

df_hh_2025 = pd.read_fwf(
    "/data/PLFS 2025/CHHV1.TXT",
    colspecs=colspecs_hh,
    names=colnames_hh
)

Pre-Processing 2025 PLFS Data

In [ ]:
# ---------------------------
# Rename columns (make 2025 consistent with 2024)
# ---------------------------
# Raw PLFS 2025 uses short names → convert to readable + consistent names

df_perv_2025.rename(columns={
    'sec':'sector',          # Rural / Urban
    'st':'state',
    'dc':'district',
    'sex':'gender',

    # Current Weekly Status variables
    'aind_cws':'industry',   # NIC code
    'acws':'worker_status',  # employment status

    'mfsu':'fsu',            # sampling unit
    'ssu':'hh_id'            # household id
}, inplace=True)

df_hh_2025.rename(columns={
    'sec':'sector',
    'st':'state',
    'dc':'district',
    'mfsu':'fsu',
    'ssu':'hh_id'
}, inplace=True)


# ---------------------------
# Merge person + household data
# ---------------------------
# 2025 data requires month + visit (panel structure)
merge_keys = ['month','visit','sector','fsu','sss','hh_id']

df_2025 = df_perv_2025.merge(
    df_hh_2025,
    on=merge_keys,
    how='left'     # keep all individuals, attach household info
)


# ---------------------------
# Clean column names after merge
# ---------------------------
# Remove suffixes (_x, _y) and keep person-level variables

df_2025.rename(columns={
    'state_x':'state',
    'district_x':'district',
    'mult_x':'mult'   # sampling weight
}, inplace=True)


# ---------------------------
# Select required columns
# ---------------------------
cols = [
    'sector','state','district',
    'gender','age','industry','worker_status','mult',
    'hh_size','hhtype','sg','relg','hce_tot'
]

# Keep only columns that exist (safe for missing fields)
cols = [c for c in cols if c in df_2025.columns]
df_2025 = df_2025[cols].copy()


# ---------------------------
# Convert data types
# ---------------------------
# Ensure numeric operations work correctly
for c in df_2025.columns:
    df_2025[c] = pd.to_numeric(df_2025[c], errors='coerce')


# ---------------------------
# Filter working population
# ---------------------------
# 11 = self-employed, 12 = salaried, 21 = casual labour
df_2025 = df_2025[df_2025['worker_status'].isin([11,12,21])]


# ---------------------------
# Drop invalid / missing records
# ---------------------------
df_2025 = df_2025[df_2025['state'].notna() & df_2025['mult'].notna()]
df_2025 = df_2025[df_2025['industry'].notna()]


# ---------------------------
# Apply survey weights
# ---------------------------
# Convert multiplier into usable weight
df_2025['weight'] = df_2025['mult'] / 100

Classifying employment sector based on industry code

In [ ]:
# ---------------------------
# Function to classify industry into broad sectors
# ---------------------------
# PLFS provides detailed NIC (industry) codes.
# For analysis, we group them into broader sectors:
# Agriculture, Manufacturing, Services, etc.

def classify_sector(nic):
    try:
        # Convert to integer (in case it's string / float)
        nic = int(nic)

        # Take first 2 digits of NIC code (industry group level)
        nic2 = int(str(nic)[:2])

        # Sector mapping based on NIC classification
        if 1 <= nic2 <= 3:
            return "Agriculture"         # Primary sector

        elif 5 <= nic2 <= 39:
            return "Manufacturing"       # Secondary sector

        elif nic2 >= 45:
            return "Services"            # Tertiary sector

        else:
            return "Other"               # Remaining categories

    except:
        # If value is invalid / missing
        return "Unknown"


# ---------------------------
# Apply classification to both years
# ---------------------------
# This creates a common comparable field across datasets

df_2024['employment_sector'] = df_2024['industry'].apply(classify_sector)
df_2025['employment_sector'] = df_2025['industry'].apply(classify_sector)


# ---------------------------
# Add year labels
# ---------------------------
# Needed for comparison and trend analysis

df_2024['year'] = 2024
df_2025['year'] = 2025


# ---------------------------
# Combine both datasets
# ---------------------------
# Creates a single dataset for analysis

df_all = pd.concat([df_2024, df_2025])

Performing the Chi Square Test

In [ ]:
# ---------------------------
# Create broad sector classification
# ---------------------------
# For hypothesis testing, we simplify sectors into:
# Agriculture vs Non-Agriculture

df_all['broad_sector'] = df_all['employment_sector'].apply(
    lambda x: 'Agriculture' if x == 'Agriculture' else 'Non-Agriculture'
)


# ---------------------------
# Build weighted contingency table
# ---------------------------
# Instead of simple counts, we use survey weights
# This ensures results reflect actual population distribution

table = (
    df_all
    .pivot_table(
        index='broad_sector',   # rows: Agriculture / Non-Agriculture
        columns='year',         # columns: 2024 / 2025
        values='weight',        # weighted population
        aggfunc='sum'
    )
)

print("Weighted Table:\n", table)


# ---------------------------
# Chi-square test of independence
# ---------------------------
# Purpose:
# Check if employment distribution (Agri vs Non-Agri)
# is significantly different between 2024 and 2025

chi2, p, dof, exp = chi2_contingency(table)

print("\np-value:", p)


# ---------------------------
# Interpret result
# ---------------------------
# If p < 0.05 → reject null hypothesis
# Meaning: there IS a statistically significant shift

if p < 0.05:
    print("Significant shift from Agriculture to Non-Agriculture")
else:
    print("No significant shift")

Saving the merged dataframe as a final CSV output for Dashboard Visualization

In [ ]:
df_all.to_csv("/data/PLFS_ODISHA_FINAL.csv", index=False)